# Model Evaluation and Deployment

This notebook evaluates the final model against a benchmark heuristic and deploys the model to SageMaker.

In [ ]:
import joblib
import pandas as pd
from sklearn.metrics import classification_report

# Load combined test data
test_df = pd.read_csv("split/Xy_test.csv")

# Adjust this column name as needed
label_column = "target"  # Replace with your actual label column name

# Split into features and labels
X_test = test_df.drop(columns=[label_column])
y_test = test_df[label_column]

# Load trained model
model = joblib.load("model-output/final_model.pkl")

# Predict
y_pred = model.predict(X_test)
print("Final Model Performance:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# Load heuristic predictions (ensure this CSV aligns with y_test order)
heuristic_preds = pd.read_csv("data/heuristic_predictions.csv").squeeze()

print("Heuristic Model Performance:\n")
print(classification_report(y_test, heuristic_preds))

## Comparison Summary

Use the above reports to compare accuracy, precision, recall, and F1-score between your trained model and the heuristic baseline.

In [ ]:
import sagemaker
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import get_execution_role

# SageMaker setup
sagemaker_session = sagemaker.Session()
role = get_execution_role()

# Update with your actual model S3 path and entry point script
model_artifact = 's3://your-bucket-name/model/final_model.tar.gz'
entry_script = 'inference.py'

# Create and deploy model
sklearn_model = SKLearnModel(
    model_data=model_artifact,
    role=role,
    entry_point=entry_script,
    framework_version='0.23-1',
    sagemaker_session=sagemaker_session
)

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name='final-model-endpoint'
)

print("Model deployed. Endpoint name: final-model-endpoint")

In [ ]:
# Test the live endpoint
sample_input = X_test.iloc[:5].values
results = predictor.predict(sample_input)
print("Sample predictions:", results)